# Project K, equal battery: OLA (7B) runs the completed Series One battery

**Why this notebook exists.** Kobby's equal-tests ruling (Nov 2,
2026): every model goes through the same Series One battery before
any post speaks across models. This model owes the whole Series One battery; the Series Two cells were its only run before this. This notebook runs the
whole battery for OLA (7B), identical to the other three
equal-battery notebooks: same files, same questions byte for byte,
same repeats, same 33-column results schema as the Series Two runs.

**The battery (41 asks).**

- **Part A (20 asks).** The repaired JSON question on the three
  real clips (vacuum, jar, eggs) and all seven swap files, two
  repeats each.
- **Part A2 (12 asks).** The same question on the three real
  clips in two built versions: muted (picture, no sound) and
  blackout (sound, no picture), two repeats each.
- **Part B (6 asks).** The plain sounds question on the three
  matched bed files (the sentence alone, the sentence over beeps,
  the sentence over the vacuum bed), two repeats each.
- **Part B door (3 asks).** A transcription question on the same
  three matched files, one repeat each. The bed is judged per
  prompt: a model's bed rows are read only through prompts whose
  baseline (the sentence alone) returned the sentence. The door
  gives every model that baseline check in its strongest form.

**What is reused, unchanged.** Step 1 and Step 3 are this model's
Series Two ones, untouched: the sealed private environment from the Series One notebook: `THUdyh/Ola-7b` at its pinned head revision, loaded by the authors' own `load_pretrained_model` on the ladder bf16, then 8-bit, then 4-bit. The runner's ask path,
feed handling, and feed proof are the Series Two ones. Only Step
2's file list and the battery are new.

**The feed proof rule.** Before every answer is taken, the runner
prints the proof for that exact feed, measured on the model's own
feed path. If a track that should carry sound decodes missing,
empty, short, or all zeros, the run hard-stops with a loud banner
and the rows so far are saved. Declared silence (the swap silence
file and the muted versions) is exempt by name in the job list and
is itself verified to be zeros. Nothing is padded quietly.

**How to run it.** Run the cells from the top, in order. Step 1
builds the environment and proves the stack. Step 2 copies the
thirteen battery files plus the sanity probe clip, builds the six muted and blackout versions,
measures them, and saves the built files into the Drive clips
folder. Step 3 writes the runner and runs its check mode (proof,
loading ladder, sanity). Step 4 runs the battery and saves
`equal_ola_results.csv` in the Drive clips folder.

## Step 1 · Setup: a private environment with the authors' stack

The next cell does six things:

1. Installs `uv`, a fast environment and package tool, if this machine
   does not already have it. That is the only thing it adds to
   Colab's own Python, and it is a tool, not part of the model's
   stack.
2. Creates a private virtual environment at `/content/ola-venv`, on
   Python 3.10, the Python the authors state, seeded with the basic
   packaging tools (one later install needs them present).
3. Installs the model's stack inside that environment, in passes.
   First, torch 2.1.2 with its matching torchvision and torchaudio,
   from PyTorch's CUDA 12.1 shelf, the build the authors pin.
   Second, the authors' own pin list: transformers 4.43.4 (the
   version stamped into the model's config), tokenizers, sentencepiece,
   accelerate, peft, bitsandbytes, timm, einops and einops-exts,
   omegaconf, loguru, av, scikit-learn, plus librosa and soundfile.
   numpy is held at 1.26.4 because torch 2.1 was built before numpy 2
   existed. Riding in the same pass are two packages the authors'
   inference code imports but their pin list forgot: decord, the
   video reader, and moviepy 1.0.3, the last moviepy release that
   still has the moviepy.editor module the code imports. Pillow
   rides along as well. One build rule rides with this pass: any
   package that has to be built from source in it is built with
   setuptools held before version 81, because openai-whisper's
   setup script imports a tool (pkg_resources) that newer
   setuptools no longer provides in the build room; without the
   cap, that one package fails the whole pass.
4. Installs the two packages OLA's code demands by bare import.
   The vision tower file imports flash_attn and deepspeed at its top
   level, so the authors' package will not even import without them.
   deepspeed goes in at the authors' version, 0.12.6, without build
   isolation, because its installer needs to see the torch from
   pass one. flash-attn goes in as Dao-AILab's official prebuilt
   wheel for this exact stack (flash-attn 2.5.8, Python 3.10, torch
   2.1, CUDA 12, the pre-cxx11 ABI that torch 2.1 ships with),
   pinned by its full download address, which was fetched and
   checked when this notebook was built. Never a build from source.
5. Clones the authors' code repo, checks that its HEAD is exactly
   the pinned commit `f681820b32db398e62d430c4e5b4f3dd962bbed4`,
   records that commit in a small file the runner will quote, and
   installs the ola package from that clone with none of its listed
   dependencies dragged in (passes one to four already placed them,
   at the pinned versions).
6. Proves the stack, in a fresh process under the environment's own
   Python, with the authors' own environment variables set first
   (their code reads them at import time). The proof prints the
   torch and transformers versions and the transformers file path,
   imports flash_attn and the two functions the vision tower calls,
   imports deepspeed, and then imports every entry point the runner
   uses from the ola package, plus decord, moviepy, and whisper. A
   version number alone is not proof; the imports are. If any line
   of the proof is wrong, Step 1 stops on the spot.

The installs download several gigabytes (torch is the big one) and
take several minutes. That is normal. Every command is printed
before it runs, and the installers stream their output into the
cell, so nothing about this step happens off the record. The full
output of every pass is also appended, as it happens, to a log
file named `ola_install_log.txt`: in the Drive clips folder when
Drive is available, and in `/content` otherwise. If a pass fails,
its STOP banner prints the installer's own last 60 lines, so the
reason is always in the open, never just a claim that it was
shown somewhere above.

In [ ]:
# --- Setup: build the private environment and prove the stack inside it --------
# This cell never installs the model's stack into Colab's Python. The
# stack lives in a private virtual environment at /content/ola-venv,
# on Python 3.10 with the authors' own pinned versions, and Step 1
# ends by proving, in a fresh process under that environment's
# Python, that the pinned versions and every import the model's code
# demands are really there. Two of those demands are bare top-level
# imports in the vision tower file (flash_attn and deepspeed): the
# authors' package refuses to import without them, so they are
# installed here as first-class members of the stack, not afterthoughts.
import os, sys, shutil, subprocess
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

VENV = "/content/ola-venv"
VENV_PY = os.path.join(VENV, "bin", "python")

LAST_OUTPUT = {}

def _install_log_paths():
    # The install log's two homes, in order: the Drive clips folder
    # when Drive is available at the time of the write, and /content
    # always. Step 1 normally runs before Drive is mounted, so the
    # /content copy is the one that exists; the Drive copy is used
    # whenever the folder is there.
    paths = []
    if os.path.isdir("/content/drive/MyDrive/project-k-clips"):
        paths.append("/content/drive/MyDrive/project-k-clips/"
                     "ola_install_log.txt")
    paths.append("/content/ola_install_log.txt")
    return paths

def _append_install_log(text):
    for p in _install_log_paths():
        try:
            with open(p, "a") as fh:
                fh.write(text)
        except Exception:
            pass   # that home is not available; the other copy stands

def run_stream(cmd, label="command"):
    # Run one install pass, streaming its output into the cell AND
    # keeping every line. The full output is appended to
    # ola_install_log.txt as the pass ends, and the text is kept in
    # LAST_OUTPUT so a failure banner can quote the installer's own
    # last 60 lines. A banner without the reason never ships.
    print("+", " ".join(cmd))
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True,
                            bufsize=1)
    lines = []
    for line in proc.stdout:
        print(line, end="")
        lines.append(line)
    proc.wait()
    print("  exit code:", proc.returncode)
    output = "".join(lines)
    LAST_OUTPUT[label] = output
    _append_install_log("=" * 70 + "\n" + label + ": "
                        + " ".join(cmd) + "\nexit code: "
                        + str(proc.returncode) + "\n" + output)
    return proc.returncode

def stop_banner(lines, exc):
    print()
    print("!" * 72)
    for ln in lines:
        print(ln)
    print("!" * 72)
    raise SystemExit(exc)

def stop_install(lines, label, exc):
    # The failure exit for an install pass: the banner carries the
    # installer's own last 60 lines and names the log file holding
    # the full output of every pass.
    tail = (LAST_OUTPUT.get(label, "") or "").splitlines()[-60:]
    print()
    print("!" * 72)
    for ln in lines:
        print(ln)
    print("--- last " + str(len(tail))
          + " lines of the installer's own output ---")
    for ln in tail:
        print(ln)
    print("--- end of installer output ---")
    print("The full output of every install pass is saved in:")
    for p in _install_log_paths():
        if os.path.exists(p):
            print("   ", p)
    print("!" * 72)
    raise SystemExit(exc)

# 1. uv: the environment tool. The only addition to Colab's own Python.
if shutil.which("uv") is None:
    print("uv not found. Installing it (a tool only, not the model stack).")
    rc = run_stream([sys.executable, "-m", "pip", "install", "uv"],
                    "uv install")
    if rc != 0 or shutil.which("uv") is None:
        stop_install(["STOP. uv could not be installed, so the private",
                      "environment cannot be built. Show this output",
                      "to Muse."], "uv install",
                     "STOP: uv install failed")
print("uv:", shutil.which("uv"))

# 2. The private environment itself, on Python 3.10, the Python the
# authors state in their own setup (their pyproject and README both
# build on 3.10, and the demanded packages, deepspeed 0.12.6 and the
# flash-attn wheel, are era matched to it). Seeded, because the
# deepspeed install in pass 3 runs without build isolation and needs
# the packaging tools present in the environment itself.
if not os.path.exists(VENV_PY):
    rc = run_stream(["uv", "venv", "--seed", "--python", "3.10", VENV],
                    "venv create")
    if rc != 0:
        stop_install(["STOP. The private environment could not be",
                      "created. Show this output to Muse."],
                     "venv create", "STOP: venv creation failed")
ver = subprocess.run([VENV_PY, "--version"], capture_output=True, text=True)
print("venv python:", (ver.stdout or ver.stderr).strip())

# 3a. torch first, at the authors' pin, from PyTorch's CUDA 12.1
# shelf: torch 2.1.2 is a CUDA 12.1 build, and taking it from the
# shelf makes that explicit instead of leaving the choice to a
# resolver that might drag in a different CUDA line.
rc = run_stream(["uv", "pip", "install", "--python", VENV_PY,
                 "torch==2.1.2", "torchvision==0.16.2", "torchaudio==2.1.2",
                 "--index-url", "https://download.pytorch.org/whl/cu121"],
                "pass 3a torch")
if rc != 0:
    stop_install(["STOP. The torch install inside the private environment",
                  "failed. Its own last lines are below, inside this",
                  "banner. Show it to Muse."], "pass 3a torch",
                 "STOP: torch install failed in venv")

# 3b. The authors' pin list, in one resolver pass, plus the ask
# path's needs. From the repo's pyproject: transformers 4.43.4 (also
# the version stamped in the model's config.json), tokenizers
# 0.19.1, sentencepiece 0.1.99, accelerate 0.33.0, peft 0.11.1,
# bitsandbytes 0.43.1, timm 0.9.16, einops 0.6.1, einops-exts 0.0.4,
# omegaconf 2.0.6, loguru, av, scikit-learn 1.2.2, openai-whisper,
# librosa, soundfile. numpy is pinned to 1.26.4 because torch 2.1
# predates numpy 2 and cannot run against it. decord 0.6.0 and
# moviepy 1.0.3 are not in the authors' list at all, but their
# inference code imports both: decord is the video reader, and
# moviepy.editor, which the code imports, exists only in moviepy
# 1.x. librosa is pinned to the patch the MiniCPM room proved.
# ninja rides along because deepspeed's own tooling expects it.
# One build rule rides with this pass, written to a small file just
# below: source builds in this pass are capped to setuptools before
# 81. openai-whisper 20240930 ships only as a source build, and its
# setup script imports pkg_resources; the isolated build room's
# current setuptools (81 and up) no longer provides that module,
# so the build, and with it this whole pass, died on it in the
# first live run. The cap constrains only the tools that build
# source packages; it changes no pin in the list itself.
CARD_PACKAGES = [
    "transformers==4.43.4",
    "tokenizers==0.19.1",
    "sentencepiece==0.1.99",
    "accelerate==0.33.0",
    "peft==0.11.1",
    "bitsandbytes==0.43.1",
    "timm==0.9.16",
    "einops==0.6.1",
    "einops-exts==0.0.4",
    "omegaconf==2.0.6",
    "loguru",
    "av",
    "scikit-learn==1.2.2",
    "numpy==1.26.4",
    "librosa==0.10.2.post1",
    "soundfile==0.13.1",
    "openai-whisper==20240930",
    "decord==0.6.0",
    "moviepy==1.0.3",
    "ninja",
    "Pillow",
]
BUILD_CONSTRAINTS = "/content/ola-build-constraints.txt"
with open(BUILD_CONSTRAINTS, "w") as fh:
    fh.write("setuptools<81\n")
rc = run_stream(["uv", "pip", "install", "--python", VENV_PY,
                 "--build-constraints", BUILD_CONSTRAINTS] + CARD_PACKAGES,
                "pass 3b authors stack")
if rc != 0:
    stop_install(["STOP. The authors' stack did not install inside the",
                  "private environment. Its own last lines are below,",
                  "inside this banner. Show it to Muse."],
                 "pass 3b authors stack",
                 "STOP: card stack install failed in venv")

# 3c-pre. pkg_resources for the deepspeed build. deepspeed's setup
# script imports torch.utils.cpp_extension, and torch 2.1's
# cpp_extension does "from pkg_resources import packaging".
# pkg_resources ships with setuptools, and setuptools 81 and up no
# longer provides it: the same removal that broke the
# openai-whisper source build in pass 3b, capped there by build
# constraints. Pass 3c builds without isolation, so its build runs
# against this environment's own setuptools, and the cap has to
# land in the environment itself. setuptools below 81 goes in
# first. Nothing in the stack needs a newer setuptools at run
# time. (Found live on Nov 2, 2026: the Series Two OLA run died
# at pass 3c on exactly this missing module.)
rc = run_stream(["uv", "pip", "install", "--python", VENV_PY,
                 "setuptools<81", "wheel"],
                "pass 3c-pre setuptools for deepspeed")
if rc != 0:
    stop_install(["STOP. The setuptools pin for the deepspeed build",
                  "did not install inside the private environment.",
                  "Its own last lines are below, inside this banner.",
                  "Show it to Muse."],
                 "pass 3c-pre setuptools for deepspeed",
                 "STOP: setuptools pin failed in venv")

# 3c. deepspeed: demanded by bare import in the vision tower file
# (ola/model/multimodal_encoder/oryx_vit.py, line 46), at the
# authors' version from their training extras, 0.12.6. It installs
# without build isolation because its installer imports torch while
# it builds, and torch is here, in this environment, from pass 3a.
# In an isolated build room it would find no torch and fail.
rc = run_stream(["uv", "pip", "install", "--python", VENV_PY,
                 "--no-build-isolation", "deepspeed==0.12.6"],
                "pass 3c deepspeed")
if rc != 0:
    stop_install(["STOP. The deepspeed install inside the private",
                  "environment failed. Its own last lines are below,",
                  "inside this banner. Show it to Muse."],
                 "pass 3c deepspeed",
                 "STOP: deepspeed install failed in venv")

# 3d. flash-attn: the other demanded import (oryx_vit.py, line 37:
# from flash_attn import flash_attn_func, flash_attn_varlen_func).
# The language model itself is loaded without flash attention (the
# authors' builder defaults use_flash_attn to False); it is the
# vision tower's own attention code that calls these functions, so
# the package has to be present and working, not merely named. It
# goes in as Dao-AILab's official prebuilt wheel, never a build from
# source: flash-attn 2.5.8 for CPython 3.10, torch 2.1, and CUDA 12,
# in the cxx11 ABI FALSE form, which is the ABI torch 2.1's own
# wheels ship with (PyTorch only moved its default builds to the
# cxx11 ABI years later). The address was fetched when this notebook
# was built: it answers HTTP 200 and the wheel is 120,616,671 bytes.
FLASH_ATTN_WHEEL = ("https://github.com/Dao-AILab/flash-attention/"
                    "releases/download/v2.5.8/flash_attn-2.5.8"
                    "%2Bcu122torch2.1cxx11abiFALSE-cp310-cp310"
                    "-linux_x86_64.whl")
rc = run_stream(["uv", "pip", "install", "--python", VENV_PY,
                 FLASH_ATTN_WHEEL], "pass 3d flash-attn")
if rc != 0:
    stop_install(["STOP. The flash-attn wheel did not install inside the",
                  "private environment. Its own last lines are below,",
                  "inside this banner. Show it to Muse."],
                 "pass 3d flash-attn",
                 "STOP: flash-attn install failed in venv")

# 3e. The authors' code itself. The hub repo ships no Python files;
# the model class lives in the ola package in their GitHub repo.
# Clone it, verify its HEAD is the pinned commit (the repo's HEAD
# on Nov 2, 2026, when this notebook was built and the code was
# read), record the commit for the runner to quote, and install the
# package with no dependencies of its own: passes 3a to 3d already
# placed every one at its pinned version.
CODE_DIR = "/content/ola-code"
CODE_COMMIT = "f681820b32db398e62d430c4e5b4f3dd962bbed4"
if not os.path.isdir(os.path.join(CODE_DIR, "ola")):
    rc = run_stream(["git", "clone", "--depth", "1",
                     "https://github.com/Ola-Omni/Ola.git", CODE_DIR],
                    "ola code clone")
    if rc != 0:
        stop_install(["STOP. The authors' code repo could not be cloned.",
                      "Show this output to Muse."], "ola code clone",
                     "STOP: ola code clone failed")
head = subprocess.run(["git", "-C", CODE_DIR, "rev-parse", "HEAD"],
                      capture_output=True, text=True)
head = (head.stdout or "").strip()
print("ola code HEAD:", head)
if head != CODE_COMMIT:
    stop_banner(["STOP. The cloned code is at commit " + head + ", not",
                 "the pinned " + CODE_COMMIT + ". The record would not",
                 "be byte-identical. Show this output to Muse."],
                "STOP: ola code commit mismatch")
with open("/content/ola-code-commit.txt", "w") as fh:
    fh.write(head + "\n")
rc = run_stream(["uv", "pip", "install", "--python", VENV_PY,
                 "--no-deps", CODE_DIR], "pass 3e ola package")
if rc != 0:
    stop_install(["STOP. The ola package did not install inside the",
                  "private environment. Its own last lines are below,",
                  "inside this banner. Show it to Muse."],
                 "pass 3e ola package",
                 "STOP: ola package install failed in venv")

# 4. Proof, in a fresh process under the environment's Python, with
# the authors' environment variables set first (their mm_utils and
# vision tower read them at import time, so the proof imports under
# the same conditions the runner will). Version alone is not proof:
# the proof imports the two functions the vision tower calls out of
# flash_attn, imports deepspeed, and imports every entry point the
# runner uses from the ola package, plus decord, moviepy, and
# whisper for the ask path.
PROOF = (
    "import os\n"
    "for k, v in {'LOWRES_RESIZE': '384x32', 'HIGHRES_BASE': '0x32',\n"
    "             'VIDEO_RESIZE': '0x64', 'VIDEO_MAXRES': '480',\n"
    "             'VIDEO_MINRES': '288', 'MAXRES': '1536', 'MINRES': '0',\n"
    "             'FORCE_NO_DOWNSAMPLE': '1', 'LOAD_VISION_EARLY': '1',\n"
    "             'PAD2STRIDE': '1'}.items():\n"
    "    os.environ[k] = v\n"
    "import torch, transformers\n"
    "print('torch version:', torch.__version__)\n"
    "print('transformers version:', transformers.__version__)\n"
    "print('transformers file:', transformers.__file__)\n"
    "import flash_attn\n"
    "from flash_attn import flash_attn_func, flash_attn_varlen_func\n"
    "print('flash_attn version:', flash_attn.__version__)\n"
    "print('flash_attn import: OK')\n"
    "import deepspeed\n"
    "print('deepspeed version:', deepspeed.__version__)\n"
    "print('deepspeed import: OK')\n"
    "from ola.model.builder import load_pretrained_model\n"
    "from ola.conversation import conv_templates\n"
    "from ola.mm_utils import process_anyres_video, KeywordsStoppingCriteria\n"
    "from ola.datasets.preprocess import tokenizer_speech_image_token\n"
    "import ola.model.builder as _b\n"
    "print('ola builder file:', _b.__file__)\n"
    "print('ola imports: OK')\n"
    "import decord, moviepy.editor, whisper\n"
    "print('ask path imports: OK')\n"
)
proof = subprocess.run([VENV_PY, "-c", PROOF],
                       capture_output=True, text=True)
print()
print(proof.stdout)
if proof.stderr.strip():
    print("proof process also said:")
    print("\n".join(proof.stderr.splitlines()[-10:]))
ok = ("transformers version: 4.43.4" in proof.stdout
      and "flash_attn import: OK" in proof.stdout
      and "deepspeed import: OK" in proof.stdout
      and "ola imports: OK" in proof.stdout
      and "ask path imports: OK" in proof.stdout)
if not ok:
    print("!" * 72)
    print("STOP. THE PROOF FAILED INSIDE THE PRIVATE ENVIRONMENT.")
    print("Either a version is wrong (transformers 4.43.4) or one of")
    print("the demanded imports failed: flash_attn, deepspeed, the ola")
    print("package entry points, or the ask path packages. Do not go")
    print("on to Step 2. Show this output to Muse.")
    print("!" * 72)
    raise SystemExit("STOP: venv proof failed")
print("proof passed: transformers 4.43.4, loaded from the path above,")
print("flash_attn and deepspeed import, and every ola entry point the")
print("runner uses imports too.")

# The chip, read from the machine itself. This cell deliberately never
# imports torch or transformers into the notebook's own session.
smi = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True)
print()
print((smi.stdout or smi.stderr).strip())
print("This notebook wants an L4 (about 24 GB). The model in full form")
print("plus its two speech encoders wants about 19 to 20 GB of chip.")
print("A T4 (about 16 GB) cannot hold it; the ladder would have to")
print("squeeze it, and the record would say so.")
print("Step 1 done. Go on to Step 2. No restart is needed.")

## Step 2 · Get the clips; build the muted and blackout versions

Everything this run asks about lives in your Google Drive, in the
folder named `project-k-clips`. The next cell links your Drive,
finds that folder, and copies thirteen battery files plus the
sanity probe clip (`probe_tone.mp4`, the beep clip the Step 3
sanity gate asks about) to a plain work folder here:

**The three real clips:**

1. `P02_07_5.mp4`: the vacuum clip.
2. `P32_01_54.mp4`: the jar clip.
3. `own-01.mp4`: your eggs recording.

**The seven swap files** (one still picture, seven soundtracks,
loudness matched at build time): `swap_silence.mp4`,
`swap_speech.mp4`, `swap_beeps.mp4`, `swap_water.mp4`,
`swap_vacuum.mp4`, `swap_sizzle.mp4`, `swap_mixed.mp4`.

**The three matched bed files:**

11. `matched_speech.mp4`: the spoken sentence alone, the baseline.
12. `matched_speech_beeps.mp4`: the sentence over a beep bed.
13. `matched_speech_vacuum.mp4`: the sentence over a vacuum bed.

If any of the fourteen is missing, the cell stops and names it.
Nothing is substituted for a missing file, the matched files
included: a bed test without its exact files is not the bed test.

**Standing rule:** a real clip longer than 20 seconds is cut to
its first 15 seconds (that is the eggs recording), the same
treatment as the earlier notebooks. The swap and matched files
are exact test files and are never cut.

**Then the cell builds six versions** from the real clips as they
stand in the work folder, and measures each one from its decoded
track before printing the measurement:

- **Muted** (`equal_muted_<name>`): the same picture, a silent
  track. The decoded track must measure as silence (RMS at or
  below -90 dB) at the source's duration, or the cell stops.
- **Blackout** (`equal_blackout_<name>`): a plain black picture
  at the clip's own size and length, the original sound
  untouched. The decoded track's duration and RMS must match the
  source clip's (within 0.5 seconds and 1.5 dB), or the cell
  stops.

Built files are saved back into the Drive clips folder, so every
model's run can ride the identical files.

Do not go on until the ready line names all of it: 3 real clips,
7 swap files, 3 matched files, 6 built versions.

In [ ]:
# --- Get the clips; build the muted and blackout versions -----------------------
# Copies thirteen battery files plus the sanity probe clip
# (probe_tone.mp4, which the runner's Step 3 sanity gate asks
# about; the Nov 2, 2026 equal run stopped at that gate because
# this list had dropped it) from the Drive clips folder, applies
# the standing cut rule to the real clips, then builds the six muted
# and blackout versions and measures every built file from its
# decoded track. A missing file stops the cell and names the
# file; a built file that decodes wrong stops the cell too.
import os, shutil, subprocess, json, math
import numpy as np

REAL = ["P02_07_5.mp4", "P32_01_54.mp4", "own-01.mp4"]
SWAP = ["swap_silence.mp4", "swap_speech.mp4", "swap_beeps.mp4",
        "swap_water.mp4", "swap_vacuum.mp4", "swap_sizzle.mp4",
        "swap_mixed.mp4"]
MATCHED = ["matched_speech.mp4", "matched_speech_beeps.mp4",
           "matched_speech_vacuum.mp4"]
PROBE = ["probe_tone.mp4"]
WORK = "/content/clips"
DRIVE_CLIPS = "/content/drive/MyDrive/project-k-clips"
os.makedirs(WORK, exist_ok=True)

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as e:
    print("Drive mount skipped:", e)

SOURCES = [DRIVE_CLIPS, "/content/project-k-clips"]

def bring(name):
    dest = os.path.join(WORK, name)
    if os.path.exists(dest):
        return True
    for folder in SOURCES:
        src = os.path.join(folder, name)
        if os.path.exists(src):
            shutil.copy(src, dest)
            print("copied", name, "from", folder)
            return True
    return False

missing = [name for name in REAL + SWAP + MATCHED + PROBE
           if not bring(name)]
if missing:
    print("!" * 72)
    print("STOP: these files are missing from the Drive clips folder:")
    for name in missing:
        print("MISSING:", name)
    print("Nothing is substituted for a missing file. Add the exact")
    print("files to the project-k-clips folder, then run this cell again.")
    print("!" * 72)
    raise SystemExit("STOP: files missing in Step 2: "
                     + ", ".join(missing))

def duration_s(path):
    out = subprocess.run(
        ["ffprobe", "-v", "error", "-show_entries", "format=duration",
         "-of", "csv=p=0", path],
        capture_output=True, text=True)
    try:
        return float(out.stdout.strip())
    except Exception:
        return None

def decode_mono(path, sr=48000):
    proc = subprocess.run(
        ["ffmpeg", "-v", "error", "-i", path, "-vn", "-ac", "1",
         "-ar", str(sr), "-f", "f32le", "-"],
        capture_output=True)
    if proc.returncode != 0:
        raise RuntimeError(f"decode failed for {path}: "
                           + proc.stderr.decode(errors="replace")[-500:])
    return np.frombuffer(proc.stdout, dtype=np.float32), sr

def rms_db(x):
    if len(x) == 0:
        return -120.0
    v = float(np.sqrt(np.mean(np.square(x, dtype=np.float64))))
    return 20.0 * math.log10(v) if v > 0 else -120.0

def save_both(local_path):
    if os.path.isdir(DRIVE_CLIPS):
        shutil.copy(local_path, os.path.join(
            DRIVE_CLIPS, os.path.basename(local_path)))
        print("saved to Drive clips folder:",
              os.path.basename(local_path))

def video_size(path):
    out = subprocess.run(
        ["ffprobe", "-v", "error", "-select_streams", "v:0",
         "-show_entries", "stream=width,height",
         "-of", "csv=p=0", path],
        capture_output=True, text=True)
    try:
        w, h = out.stdout.strip().split(",")
        return int(w), int(h)
    except Exception:
        return None, None

# The standing cut rule, real clips only: longer than 20 seconds
# means cut to the first 15. Swap and matched files are exact
# test files and are never cut.
for name in REAL:
    path = os.path.join(WORK, name)
    d = duration_s(path)
    if d is not None and d > 20:
        short = os.path.join(WORK, "short_" + name)
        subprocess.run(["ffmpeg", "-y", "-v", "error", "-i", path,
                        "-t", "15", "-c:v", "libx264",
                        "-pix_fmt", "yuv420p", "-c:a", "aac", short],
                       check=False)
        if os.path.exists(short):
            os.replace(short, path)
            print("cut", name, "to its first 15 seconds")

build_log = []

# ---- muted versions -------------------------------------------------------------
for name in REAL:
    src = os.path.join(WORK, name)
    out = os.path.join(WORK, "equal_muted_" + name)
    if not os.path.exists(out):
        proc = subprocess.run(
            ["ffmpeg", "-y", "-v", "error", "-i", src,
             "-f", "lavfi", "-i",
             "anullsrc=channel_layout=stereo:sample_rate=48000",
             "-map", "0:v:0", "-map", "1:a:0",
             "-c:v", "copy", "-c:a", "aac", "-ar", "48000",
             "-shortest", out])
        if proc.returncode != 0 or not os.path.exists(out):
            raise SystemExit(f"STOP: muted build failed for {name}")
    x, sr = decode_mono(out)
    entry = {"file": os.path.basename(out), "kind": "muted",
             "duration_s": round(len(x) / sr, 3),
             "audio_rms_db_decoded": round(rms_db(x), 2),
             "nonzero_share": round(float(np.count_nonzero(x) / len(x)), 4)}
    src_d = duration_s(src)
    if (src_d is None or abs(entry["duration_s"] - src_d) > 0.5
            or entry["audio_rms_db_decoded"] > -90.0):
        raise SystemExit(f"STOP: muted build for {name} decodes "
                         f"wrong: {entry}")
    build_log.append(entry)
    save_both(out)
    print("muted:", entry)

# ---- blackout versions ------------------------------------------------------------
for name in REAL:
    src = os.path.join(WORK, name)
    out = os.path.join(WORK, "equal_blackout_" + name)
    w, h = video_size(src)
    d = duration_s(src)
    if w is None or d is None:
        raise SystemExit(f"STOP: cannot read size or duration of {name}")
    if not os.path.exists(out):
        proc = subprocess.run(
            ["ffmpeg", "-y", "-v", "error", "-f", "lavfi",
             "-i", f"color=c=black:s={w}x{h}:r=25:d={d:.3f}",
             "-i", src, "-map", "0:v", "-map", "1:a",
             "-t", f"{d:.3f}", "-c:v", "libx264",
             "-pix_fmt", "yuv420p", "-c:a", "aac", "-ar", "48000",
             "-shortest", out])
        if proc.returncode != 0 or not os.path.exists(out):
            raise SystemExit(f"STOP: blackout build failed for {name}")
    xs, srs = decode_mono(src)
    x, sr = decode_mono(out)
    entry = {"file": os.path.basename(out), "kind": "blackout",
             "duration_s": round(len(x) / sr, 3),
             "audio_rms_db_decoded": round(rms_db(x), 2),
             "source_rms_db_decoded": round(rms_db(xs), 2),
             "nonzero_share": round(float(np.count_nonzero(x) / len(x)), 4)}
    if (abs(entry["duration_s"] - d) > 0.5
            or abs(entry["audio_rms_db_decoded"]
                   - entry["source_rms_db_decoded"]) > 1.5):
        raise SystemExit(f"STOP: blackout build for {name} decodes "
                         f"wrong: {entry}")
    build_log.append(entry)
    save_both(out)
    print("blackout:", entry)

if build_log:
    log_path = os.path.join(WORK, "equal_build_log.json")
    with open(log_path, "w") as fh:
        json.dump(build_log, fh, indent=2)
    if os.path.isdir(DRIVE_CLIPS):
        shutil.copy(log_path, os.path.join(DRIVE_CLIPS,
                                           "equal_build_log.json"))
    print("builds logged:", log_path)

print()
ready = True
for name in REAL + SWAP + MATCHED:
    path = os.path.join(WORK, name)
    if os.path.exists(path):
        print("ready:  ", name, f"({os.path.getsize(path) / 1e6:.1f} MB)")
    else:
        ready = False
        print("MISSING:", name)
for name in REAL:
    for kind in ("muted", "blackout"):
        path = os.path.join(WORK, f"equal_{kind}_" + name)
        if os.path.exists(path):
            print("ready:  ", f"equal_{kind}_" + name,
                  f"({os.path.getsize(path) / 1e6:.1f} MB)")
        else:
            ready = False
            print("MISSING:", f"equal_{kind}_" + name)
if not ready:
    print()
    print("!" * 72)
    print("STOP. Some files are missing. Check the Drive folder name is")
    print("exactly project-k-clips, then run this cell again.")
    print("Do not go on to Step 3.")
    print("!" * 72)
    raise SystemExit("STOP: files missing in Step 2")
print()
print("ready: 3 real clips, 7 swap files, 3 matched files, "
      "6 built versions (3 muted, 3 blackout)")

## Step 3 · Write the runner, then check: proof, ladder, sanity

The model never runs inside this notebook's cells directly. It
runs in a plain script, `/content/ola_equal_run.py`, under the private environment Step 1 built, through its own Python (VENV_PY).
The first cell below writes that script. The second runs it in
**check mode**.

Check mode does three things, in order:

1. **Proof again.** The versions and files the runner imported,
   with the stack pins checked a second time, exactly as the
   Series Two runner proved them.
2. **The loading ladder.** the sealed private environment from the Series One notebook: `THUdyh/Ola-7b` at its pinned head revision, loaded by the authors' own `load_pretrained_model` on the ladder bf16, then 8-bit, then 4-bit. A failed rung prints its
   full traceback, never a cut-short reason, and appends the same
   words to `equal_ola_load_log.txt` in the clips folder, so a verbatim record
   survives the session. Whichever rung loads is printed and
   saved with every answer.
3. **The sanity question.** The Series Two gate in its own form:
   one plain text question through the real ask path, and the
   word ready is required in the answer. If the answer comes
   back empty, without the word, or with an error, the script
   stops and Step 4 must not be run. Sound arrival is not this
   gate's job; Step 4's per-ask feed proof carries that.

The ladder, the sanity gate, and the ask path are this model's
Series Two ones, unchanged. Only the battery in Step 4 is new.

In [ ]:
# --- Write the equal battery runner script ----------------------------------------
# The runner holds the download, the ladder, the sanity check,
# the strict feed proof, and the equal battery. It is
# written to a plain file here and only ever run under the
# private environment's Python, never imported into this
# session.
RUNNER_SOURCE = '\n# ola_equal_run.py - the equal battery runner for OLA (7B).\n# Written by project_k_ola_equal_colab.ipynb. Run only under the\n# private environment\'s Python (/content/ola-venv/bin/python).\n# Modes: "check" runs the proof, the model download, the loading\n# ladder, and the sanity question, then stops. "battery" does all\n# of that and then runs the equal battery tests and saves\n# the CSV.\nimport os\n\n# The authors\' environment variables, set before anything of theirs\n# is imported: ola/mm_utils.py and the vision tower read them at\n# import time. These are the values inference/infer.py sets, word\n# for word.\nfor _k, _v in {\'LOWRES_RESIZE\': \'384x32\', \'HIGHRES_BASE\': \'0x32\',\n               \'VIDEO_RESIZE\': \'0x64\', \'VIDEO_MAXRES\': \'480\',\n               \'VIDEO_MINRES\': \'288\', \'MAXRES\': \'1536\', \'MINRES\': \'0\',\n               \'FORCE_NO_DOWNSAMPLE\': \'1\', \'LOAD_VISION_EARLY\': \'1\',\n               \'PAD2STRIDE\': \'1\'}.items():\n    os.environ[_k] = _v\nos.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")\nimport sys\n\nMODE = sys.argv[1] if len(sys.argv) > 1 else "check"\nprint("runner mode:", MODE)\nprint("runner python:", sys.version.split()[0], sys.executable)\n\n# --- Proof lines, printed before anything loads --------------------------------\n# The same proof Step 1 ran, repeated here so every run of this\n# script carries its own evidence: the versions it actually\n# imported, from the paths it actually loaded them, and the imports\n# the model\'s code demands by name.\nimport torch, transformers\nprint("torch version:", torch.__version__)\nprint("transformers version:", transformers.__version__)\nprint("transformers file:", transformers.__file__)\nimport flash_attn\nfrom flash_attn import flash_attn_func, flash_attn_varlen_func\nprint("flash_attn version:", flash_attn.__version__)\nprint("flash_attn import: OK")\nimport deepspeed\nprint("deepspeed version:", deepspeed.__version__)\nprint("deepspeed import: OK")\nfrom ola.model.builder import load_pretrained_model\nimport ola.model.builder as _ola_builder\nprint("ola builder file:", _ola_builder.__file__)\nprint("ola imports: OK")\nif transformers.__version__ != "4.43.4":\n    print("STOP: this script is running on transformers",\n          transformers.__version__, "and needs exactly 4.43.4, the",\n          "version the authors pin and the model config stamps.")\n    sys.exit(3)\n\n# The yes-score question, settled from the model\'s own code before\n# the run (ola/model/language_model/ola_qwen.py in the pinned code\n# commit): OlaQwenForCausalLM.generate prepares the multimodal\n# embeddings itself, then calls super().generate with every\n# remaining keyword forwarded. The standard generate therefore\n# accepts output_scores and return_dict_in_generate, and those\n# flags only add the per-step scores to the same greedy call\'s\n# return; they cannot change the prompt, the feed, or the tokens\n# chosen. MiniCPM\'s chat door, by contrast, returns decoded text\n# only, which is why its Series Two rows read "not exposed". Here\n# the score is exposed by the door itself, so YES_SCORE_EXPOSED is\n# True, and yes/no asks record the probability of a yes token at\n# the first generated position from that call\'s first score row.\nYES_SCORE_EXPOSED = True\nprint("yes token score exposed by the generate door:", YES_SCORE_EXPOSED,\n      "(generate is called directly here, as in the authors\' inference",\n      "entry, and its own scores are returned on request)")\n\n# --- The model files: one pinned revision, downloaded once ----------------------\n# The hub repo is downloaded at one exact revision into a plain\n# local folder. The config inside it points at the two speech\n# encoder files with relative paths (./large-v3.pt and the BEATs\n# checkpoint, both of which ship inside the repo). The model card\'s\n# own step 2 says to replace those paths with the local ones; the\n# runner does exactly that, in this downloaded copy, by absolute\n# path, and prints what it wrote.\nimport json\nfrom huggingface_hub import snapshot_download\n\nREPO = "THUdyh/Ola-7b"\nREVISION = "d2fcaa533792480ce748321bea5101fed29d857e"\nMODEL_DIR = "/content/ola-model"\n\nif not os.path.exists(os.path.join(MODEL_DIR, "config.json")):\n    snapshot_download(repo_id=REPO, revision=REVISION, local_dir=MODEL_DIR)\n    print("downloaded:", REPO, "at pinned revision", REVISION)\nelse:\n    print("model dir already present:", MODEL_DIR)\n_cfg_path = os.path.join(MODEL_DIR, "config.json")\nwith open(_cfg_path) as _fh:\n    _cfg = json.load(_fh)\n_cfg["speech_encoder"] = os.path.join(MODEL_DIR, "large-v3.pt")\n_cfg["music_encoder"] = os.path.join(\n    MODEL_DIR, "BEATs_iter3_plus_AS2M_finetuned_on_AS2M_cpt2.pt")\nwith open(_cfg_path, "w") as _fh:\n    json.dump(_cfg, _fh, indent=2)\nprint("config speech_encoder:", _cfg["speech_encoder"])\nprint("config music_encoder:", _cfg["music_encoder"])\n\n# --- The load log ----------------------------------------------------------------\n# A failed rung prints its full traceback (never a cut-short\n# reason) and the same words are appended, as events happen, to\n# equal_ola_load_log.txt, in the Drive clips folder first and\n# /content second, so the file is already on disk even if the\n# session dies right after a rung does.\nimport traceback, datetime, gc\n\nLOAD_LOG_PATHS = ["/content/drive/MyDrive/project-k-clips/equal_ola_load_log.txt",\n                  "/content/equal_ola_load_log.txt"]\n\ndef log_load(text):\n    for p in LOAD_LOG_PATHS:\n        try:\n            with open(p, "a") as fh:\n                fh.write(text)\n        except Exception:\n            pass   # that home is not mounted; the printed copy stands\n\ndef log_paths_saved():\n    return [p for p in LOAD_LOG_PATHS if os.path.exists(p)]\n\nCODE_COMMIT = "(not recorded)"\ntry:\n    with open("/content/ola-code-commit.txt") as _fh:\n        CODE_COMMIT = _fh.read().strip()\nexcept Exception:\n    pass\n\nlog_load("=" * 70 + "\\n")\nlog_load("OLA Series Two load log - "\n         + datetime.datetime.now().isoformat(timespec="seconds")\n         + " - runner mode: " + MODE + "\\n")\nlog_load("repo: " + REPO + " at pinned revision " + REVISION + "\\n")\nlog_load("code: github.com/Ola-Omni/Ola at commit " + CODE_COMMIT + "\\n")\nprint("ola code commit for this run:", CODE_COMMIT)\n\n# --- Load the model: the ladder, through the authors\' own builder ----------------\n# Three rungs, all of them the same model, THUdyh/Ola-7b, loaded by\n# the authors\' load_pretrained_model, exactly as the Series One\n# OLA runner loads it. Rung 1 is full form, 16-bit (bfloat16), the\n# authors\' default, finished the way their inference script\n# finishes it (to cuda, eval, bfloat16). Rungs 2 and 3 are the\n# builder\'s own squeeze flags, load_8bit and load_4bit, each with\n# device_map passed through to the loader so the squeezed weights\n# land on the chip directly. If every rung fails, the run stops\n# here and its closing lines name the log file: no substitute\n# model is ever loaded quietly.\ndef free_memory():\n    gc.collect()\n    if torch.cuda.is_available():\n        torch.cuda.empty_cache()\n\ntokenizer = None\nmodel = None\nimage_processor = None\nMODEL_NAME = None   # repo name of the model that loaded\nLOAD_MODE = None    # the fit that worked: bf16, 8-bit, or 4-bit\n\ndef try_load(mode):\n    from ola.model.builder import load_pretrained_model\n    from ola.utils import disable_torch_init\n    disable_torch_init()\n    if mode == "bf16":\n        tok, m, proc, _ctx = load_pretrained_model(MODEL_DIR, None)\n        m = m.to("cuda").eval()\n        m = m.bfloat16()\n    elif mode == "8-bit":\n        tok, m, proc, _ctx = load_pretrained_model(\n            MODEL_DIR, None, load_8bit=True, device_map="cuda")\n        m = m.eval()\n    elif mode == "4-bit":\n        tok, m, proc, _ctx = load_pretrained_model(\n            MODEL_DIR, None, load_4bit=True, device_map="cuda")\n        m = m.eval()\n    return tok, m, proc\n\nrung_errors = []\nfor mode in ("bf16", "8-bit", "4-bit"):\n    print("=" * 70)\n    print(f"rung: {REPO} in {mode} form")\n    try:\n        t, m, proc = try_load(mode)\n        tokenizer, model, image_processor = t, m, proc\n        MODEL_NAME, LOAD_MODE = REPO, mode\n        print("loaded:", REPO, "in", mode, "form")\n        break\n    except Exception:\n        tb = traceback.format_exc()\n        rung_errors.append((REPO, mode, tb))\n        print("this rung failed. Full traceback:")\n        print(tb)\n        log_load("-" * 70 + "\\n")\n        log_load("rung failed: " + REPO + " in " + mode + " form\\n")\n        log_load(tb)\n        tokenizer = None\n        model = None\n        image_processor = None\n        free_memory()\n\nprint()\nif model is None:\n    print("!" * 70)\n    print("ALL THREE RUNGS FAILED. OLA did not load in any fit.")\n    saved = log_paths_saved()\n    if saved:\n        print("The full traceback of every rung is saved, word for word, in:")\n        for p in saved:\n            print("   ", p)\n    else:\n        print("The load log could not be written anywhere; the full")\n        print("tracebacks are in the output above, and nowhere else.")\n    print("Do not go on. Show this whole output to Muse.")\n    print("!" * 70)\n    sys.exit(4)\nprint("MODEL FOR THIS RUN:", MODEL_NAME, "in", LOAD_MODE, "form")\nlog_load("loaded: " + MODEL_NAME + " in " + LOAD_MODE + " form\\n")\nprint("environment: private venv /content/ola-venv, python 3.10,",\n      "torch 2.1.2+cu121, the authors\' pinned stack")\nif torch.cuda.is_available():\n    used = torch.cuda.memory_allocated() / 1e9\n    print(f"chip memory already used by the model: {used:.1f} GB")\n\n# The yes token ids, read from the model\'s own tokenizer: the\n# first token of "Yes" and of " Yes". A yes score is the summed\n# probability of those ids at the first generated position.\nYES_TOKEN_IDS = []\nfor _word in ("Yes", " Yes"):\n    _enc = tokenizer.encode(_word, add_special_tokens=False)\n    if _enc:\n        YES_TOKEN_IDS.append(int(_enc[0]))\nYES_TOKEN_IDS = sorted(set(YES_TOKEN_IDS))\nprint("yes token ids for scoring:", YES_TOKEN_IDS,\n      [tokenizer.decode([i]) for i in YES_TOKEN_IDS])\n\n# --- Feed proof machinery ---------------------------------------------------------\n# The audit rule: the proof is printed from inside the session, per\n# ask, before the answer is taken, and it measures what OLA\'s own\n# path actually extracted. The audio numbers describe the exact\n# waveform the authors\' loader decodes: the moviepy-written wav,\n# loaded at 16 kHz with the same call the loader itself makes,\n# before its 30 second chunking pads anything. The chunk is OLA\'s\n# own unit (480000 samples, 30 s at 16 kHz), so the CSV\'s unit\n# columns carry per-chunk numbers in this run: each chunk\'s RMS\n# over its real samples, the padding that chunk needs to reach\n# 480000, and how many chunks are all zeros. The hard-stop\n# thresholds are the MiniCPM Series Two runner\'s: a track that\n# should carry sound decoding missing, empty, short by more than\n# 0.5 s against the file\'s own duration, or all zeros stops the\n# run; declared silence is exempt by name and verified to be zeros.\nimport csv, hashlib, math, re, shutil, subprocess\nimport numpy as np\nimport librosa\nimport whisper\nimport moviepy.editor as mp\nfrom decord import VideoReader, cpu\nfrom PIL import Image\nfrom ola.conversation import conv_templates, SeparatorStyle\nfrom ola.datasets.preprocess import (tokenizer_speech_image_token,\n                                     tokenizer_speech_token)\nfrom ola.mm_utils import KeywordsStoppingCriteria, process_anyres_video\nfrom ola.constants import (DEFAULT_IMAGE_TOKEN, IMAGE_TOKEN_INDEX,\n                           DEFAULT_SPEECH_TOKEN, SPEECH_TOKEN_INDEX)\n\nCLIPS = "/content/clips"\nAUDIO_DIR = "/content/ola_audio"\nos.makedirs(AUDIO_DIR, exist_ok=True)\nTRACK_DIRS = ["/content/drive/MyDrive/project-k-clips/equal_ola_extracted_tracks",\n              "/content/equal_ola_extracted_tracks"]\nfor d in TRACK_DIRS:\n    try:\n        os.makedirs(d, exist_ok=True)\n    except Exception:\n        pass\n\nCHUNK_SAMPLES = 480000   # the authors\' 30 s chunk at 16 kHz\n\nclass FeedProofError(RuntimeError):\n    pass\n\ndef rms_db(x):\n    if len(x) == 0:\n        return -120.0\n    v = float(np.sqrt(np.mean(np.square(x, dtype=np.float64))))\n    return 20.0 * math.log10(v) if v > 0 else -120.0\n\ndef media_duration(path):\n    out = subprocess.run(\n        ["ffprobe", "-v", "error", "-show_entries", "format=duration",\n         "-of", "csv=p=0", path], capture_output=True, text=True)\n    try:\n        return float(out.stdout.strip())\n    except Exception:\n        return None\n\ndef file_sha256(path):\n    h = hashlib.sha256()\n    with open(path, "rb") as fh:\n        for block in iter(lambda: fh.read(1 << 20), b""):\n            h.update(block)\n    return h.hexdigest()\n\ndef load_audio(audio_file_name):\n    # The authors\' loader, as shipped in inference/infer.py: 16 kHz\n    # mono, the sound cut into 30 second chunks (480000 samples),\n    # each chunk padded or trimmed to length, whisper log-mel with\n    # 128 mels per chunk, at most 25 chunks kept. The raw waveform\n    # chunks ride along because the BEATs half of the dual encoder\n    # reads them.\n    speech_wav, samplerate = librosa.load(audio_file_name, sr=16000)\n    if len(speech_wav.shape) > 1:\n        speech_wav = speech_wav[:, 0]\n    speech_wav = speech_wav.astype(np.float32)\n    CHUNK_LIM = 480000\n    speechs = []\n    speech_wavs = []\n    if len(speech_wav) <= CHUNK_LIM:\n        speech = whisper.pad_or_trim(speech_wav)\n        speech_wav = whisper.pad_or_trim(speech_wav)\n        speechs.append(speech)\n        speech_wavs.append(torch.from_numpy(speech_wav).unsqueeze(0))\n    else:\n        for i in range(0, len(speech_wav), CHUNK_LIM):\n            chunk = speech_wav[i : i + CHUNK_LIM]\n            if len(chunk) < CHUNK_LIM:\n                chunk = whisper.pad_or_trim(chunk)\n            speechs.append(chunk)\n            speech_wavs.append(torch.from_numpy(chunk).unsqueeze(0))\n    mels = []\n    for chunk in speechs:\n        chunk = whisper.log_mel_spectrogram(chunk, n_mels=128).permute(1, 0).unsqueeze(0)\n        mels.append(chunk)\n    mels = torch.cat(mels, dim=0)\n    speech_wavs = torch.cat(speech_wavs, dim=0)\n    if mels.shape[0] > 25:\n        mels = mels[:25]\n        speech_wavs = speech_wavs[:25]\n    speech_length = torch.LongTensor([mels.shape[1]] * mels.shape[0])\n    speech_chunks = torch.LongTensor([mels.shape[0]])\n    return mels, speech_length, speech_chunks, speech_wavs\n\ndef extract_track(path):\n    # The extraction both of the authors\' doors use: moviepy takes\n    # the sound out of the video file itself and writes it as a\n    # wav. Any failure here is a feed failure, named as one.\n    tag = re.sub(r"[^A-Za-z0-9]+", "_", os.path.basename(path))\n    wav_path = os.path.join(AUDIO_DIR, tag + ".wav")\n    try:\n        clip = mp.VideoFileClip(path)\n        try:\n            if clip.audio is None:\n                raise FeedProofError(\n                    "moviepy found no audio track in " + path)\n            clip.audio.write_audiofile(wav_path, logger=None)\n        finally:\n            clip.close()\n    except FeedProofError:\n        raise\n    except Exception as e:\n        raise FeedProofError(f"audio extraction failed for {path}: {e}")\n    if not os.path.exists(wav_path) or os.path.getsize(wav_path) == 0:\n        raise FeedProofError("moviepy wrote no audio file for " + path)\n    return wav_path\n\ndef prove_track(wav_path, media_path, expected_sound, expected_silence):\n    # Measure the waveform the authors\' loader will decode, from\n    # the wav the extraction just wrote, with the loader\'s own\n    # decode settings (librosa, 16 kHz, mono). Raises\n    # FeedProofError on any failed check; returns the proof dict\n    # on success. The frame and feed fields are filled in by the\n    # ask functions, which know which door is being ridden.\n    audio_np, sr = librosa.load(wav_path, sr=16000, mono=True)\n    audio_np = np.asarray(audio_np, dtype=np.float32)\n    if audio_np.ndim > 1:\n        audio_np = audio_np[:, 0]\n    src_dur = media_duration(media_path)\n    if src_dur is None:\n        raise FeedProofError(\n            f"could not read media duration for {media_path}")\n    track_dur = len(audio_np) / sr if sr else 0.0\n    if len(audio_np) == 0:\n        raise FeedProofError(f"extracted track is empty for {media_path}")\n    if track_dur < src_dur - 0.5:\n        raise FeedProofError(\n            f"extracted track is short for {media_path}: "\n            f"{track_dur:.3f}s of track for a {src_dur:.3f}s file")\n    overall_rms = round(rms_db(audio_np), 2)\n    nonzero_share = float(np.count_nonzero(audio_np) / len(audio_np))\n    n_chunks = int(math.ceil(len(audio_np) / CHUNK_SAMPLES))\n    if n_chunks <= 0:\n        n_chunks = 1\n    per_chunk_rms = []\n    per_chunk_padding = []\n    zero_chunks = 0\n    for i in range(n_chunks):\n        seg = audio_np[i * CHUNK_SAMPLES:(i + 1) * CHUNK_SAMPLES]\n        per_chunk_rms.append(round(rms_db(seg), 2))\n        per_chunk_padding.append(int(CHUNK_SAMPLES - len(seg)))\n        if len(seg) and not np.any(seg):\n            zero_chunks += 1\n    if expected_sound and (nonzero_share == 0.0 or overall_rms <= -119.5\n                           or zero_chunks == n_chunks):\n        raise FeedProofError(\n            f"track that should carry sound decoded silent for "\n            f"{media_path}: rms={overall_rms} dB, "\n            f"nonzero share={nonzero_share}")\n    if expected_silence and nonzero_share > 0.001:\n        raise FeedProofError(\n            f"declared silence track is not silent for {media_path}: "\n            f"nonzero share={nonzero_share}")\n    track_hash = hashlib.sha256(audio_np.tobytes()).hexdigest()\n    track_name = (re.sub(r"[^A-Za-z0-9]+", "_", os.path.basename(media_path))\n                  + "__" + track_hash[:12] + ".wav")\n    track_paths = []\n    for d in TRACK_DIRS:\n        try:\n            tp = os.path.join(d, track_name)\n            if not os.path.exists(tp):\n                shutil.copy(wav_path, tp)\n            track_paths.append(tp)\n        except Exception:\n            pass\n    proof = {\n        "file": os.path.basename(media_path),\n        "source_file_sha256": file_sha256(media_path),\n        "frame_ffmpeg_rc": ("n/a (decord reads frames in-process; "\n                            "OLA\'s path makes no ffmpeg call)"),\n        "audio_ffmpeg_rc": ("n/a (moviepy extracts in-process; "\n                            "OLA\'s path makes no ffmpeg call)"),\n        "media_duration_s": round(src_dur, 3),\n        "extracted_wav_path": wav_path,\n        "track_duration_s": round(track_dur, 3),\n        "sample_rate": int(sr),\n        "sample_count": int(len(audio_np)),\n        "track_rms_db": overall_rms,\n        "track_nonzero_share": round(nonzero_share, 6),\n        "unit_count": int(n_chunks),\n        "per_unit_rms_db": per_chunk_rms,\n        "per_unit_padding_samples": per_chunk_padding,\n        "padding_samples_total": int(sum(per_chunk_padding)),\n        "zero_chunk_count": int(zero_chunks),\n        "track_sha256": track_hash,\n        "track_copy_paths": track_paths,\n        "expected_sound": bool(expected_sound),\n        "expected_silence": bool(expected_silence),\n    }\n    return proof\n\n# The feeds, stated exactly. The video text is the Series One OLA\n# feed note, extended with the file\'s total frame count and the\n# per-ask token cap. The sound-alone text states the authors\'\n# audio+text door and the placeholder fact, with its code address.\nFEED_VIDEO_TEMPLATE = (\n    "authors\' video path (Ola repo, inference/infer.py): the "\n    "clip\'s own sound is taken out of the video by moviepy and "\n    "decoded at 16 kHz; whisper log-mel chunks of 30 s (128 mels) "\n    "feed the Whisper large-v3 half of the dual speech encoder "\n    "and the raw waveform feeds the BEATs half; decord samples "\n    "64 frames evenly over the whole clip and each frame goes "\n    "through process_anyres_video at the authors\' resize "\n    "settings (VIDEO_MAXRES 480, VIDEO_MINRES 288); the authors\' "\n    "own frame selection check then kept {kept} of the {sampled} "\n    "sampled frames (of {total} frames in the file), and that "\n    "kept set is the picture the model saw; the question rides "\n    "after the speech token and the image token in the qwen_1_5 "\n    "template; greedy decoding (temperature 0, sampling off), "\n    "at most {tokens} new tokens")\nFEED_AUDIO_TEMPLATE = (\n    "authors\' audio+text path (Ola repo, inference/infer.py, the "\n    "branch taken when an audio file is given with no image and "\n    "no video; the script names that modality \'text\'): the "\n    "soundtrack is taken out of the same file by the same "\n    "moviepy step the video door uses, then loaded alone; "\n    "whisper log-mel chunks of 30 s (128 mels) feed the Whisper "\n    "large-v3 half of the dual speech encoder and the raw "\n    "waveform feeds the BEATs half; the question rides as text "\n    "after the speech token, with no image token in the prompt, "\n    "tokenized by the authors\' speech tokenizer, and generate "\n    "is called with modalities=[\'text\']; the blank zero image "\n    "tensor that branch passes as the vision placeholder is "\n    "encoded by the vision tower but contributes no tokens to "\n    "the sequence, because the prompt holds no image token "\n    "(ola/model/ola_arch.py appends its features as an empty "\n    "slice in that case); greedy decoding (temperature 0, "\n    "sampling off), at most {tokens} new tokens")\n\ndef yes_probability(scores):\n    # The probability of a yes token at the first generated\n    # position, from the first score row of the same generate\n    # call that produced the answer. Softmax over the vocabulary,\n    # summed over the yes token ids read from the model\'s own\n    # tokenizer at load time.\n    if not scores:\n        return None\n    logits = scores[0][0].float()\n    probs = torch.softmax(logits, dim=-1)\n    total = 0.0\n    for i in YES_TOKEN_IDS:\n        total += float(probs[i].item())\n    return total\n\ndef run_generate(input_ids, media_kwargs, conv, max_new_tokens,\n                 want_scores):\n    # One generate call through the model\'s own generate method,\n    # greedy, with the authors\' stopping criteria, exactly as both\n    # of the authors\' doors shape it. On yes/no asks the call is\n    # also asked to return its per-step scores; that flag adds\n    # numbers to the return and changes nothing else.\n    attention_masks = input_ids.ne(151643).long().to("cuda")\n    stop_str = conv.sep if conv.sep_style != SeparatorStyle.TWO else conv.sep2\n    stopping_criteria = KeywordsStoppingCriteria([stop_str], tokenizer,\n                                                 input_ids)\n    kwargs = dict(inputs=input_ids,\n                  attention_mask=attention_masks,\n                  use_cache=True,\n                  stopping_criteria=[stopping_criteria],\n                  do_sample=False,\n                  num_beams=1,\n                  max_new_tokens=max_new_tokens)\n    kwargs.update(media_kwargs)\n    if want_scores:\n        kwargs["return_dict_in_generate"] = True\n        kwargs["output_scores"] = True\n    with torch.inference_mode():\n        out = model.generate(**kwargs)\n    if want_scores:\n        output_ids = out.sequences\n        scores = out.scores\n    else:\n        output_ids = out\n        scores = None\n    text = tokenizer.batch_decode(output_ids, skip_special_tokens=True)[0]\n    text = text.strip()\n    if text.endswith(stop_str):\n        text = text[:-len(stop_str)]\n    text = text.strip()\n    return text, scores\n\ndef speech_tensors(wav_path):\n    speech, speech_length, speech_chunk, speech_wav = load_audio(wav_path)\n    return ([speech.bfloat16().to("cuda")],\n            [speech_length.to("cuda")],\n            [speech_chunk.to("cuda")],\n            [speech_wav.to("cuda")])\n\ndef ask_video(path, prompt, max_new_tokens, want_yes,\n              expected_sound, expected_silence):\n    # One question about one clip, through the authors\' video\n    # door. The feed proof prints before the answer is taken.\n    if not os.path.exists(path):\n        raise FeedProofError(f"feed file missing: {path}")\n    wav_path = extract_track(path)\n    proof = prove_track(wav_path, path, expected_sound, expected_silence)\n    speechs, speech_lengths, speech_chunks, speech_wavs = speech_tensors(\n        wav_path)\n    try:\n        vr = VideoReader(path, ctx=cpu(0))\n        total_frame_num = len(vr)\n        frame_idx = np.linspace(0, total_frame_num - 1, 64,\n                                dtype=int).tolist()\n        spare_frames = vr.get_batch(frame_idx).asnumpy()\n    except FeedProofError:\n        raise\n    except Exception as e:\n        raise FeedProofError(f"frame reading failed for {path}: {e}")\n    video = [Image.fromarray(f) for f in spare_frames]\n    qs = DEFAULT_SPEECH_TOKEN + DEFAULT_IMAGE_TOKEN + "\\n" + prompt\n    conv = conv_templates["qwen_1_5"].copy()\n    conv.append_message(conv.roles[0], qs)\n    conv.append_message(conv.roles[1], None)\n    full_prompt = conv.get_prompt()\n    input_ids = tokenizer_speech_image_token(\n        full_prompt, tokenizer, IMAGE_TOKEN_INDEX,\n        return_tensors="pt").unsqueeze(0).to("cuda")\n    # Frames through the authors\' video processing, with their\n    # selection check kept exactly as shipped, as in the Series\n    # One runner: a sampled frame is kept only if its position in\n    # the sampled list also appears in the sampled index list.\n    video_processed = []\n    for idx, frame in enumerate(video):\n        image_processor.do_resize = False\n        image_processor.do_center_crop = False\n        frame = process_anyres_video(frame, image_processor)\n        if frame_idx is not None and idx in frame_idx:\n            video_processed.append(frame.unsqueeze(0))\n        elif frame_idx is None:\n            video_processed.append(frame.unsqueeze(0))\n    if not video_processed:\n        raise FeedProofError(\n            "the authors\' frame selection kept no frames for " + path)\n    video_processed = torch.cat(video_processed, dim=0).bfloat16().to("cuda")\n    kept = int(video_processed.shape[0])\n    proof["frames_total"] = int(total_frame_num)\n    proof["frames_sampled"] = int(len(video))\n    proof["frames_kept"] = kept\n    proof["feed"] = FEED_VIDEO_TEMPLATE.format(\n        kept=kept, sampled=len(video), total=int(total_frame_num),\n        tokens=max_new_tokens)\n    print("FEED PROOF " + json.dumps(proof, sort_keys=True))\n    text, scores = run_generate(\n        input_ids,\n        dict(images=video_processed, images_highres=video_processed,\n             modalities="video", speech=speechs,\n             speech_lengths=speech_lengths, speech_chunks=speech_chunks,\n             speech_wav=speech_wavs),\n        conv, max_new_tokens, want_yes)\n    yp = yes_probability(scores) if want_yes else None\n    if torch.cuda.is_available():\n        torch.cuda.empty_cache()\n    return text, proof, yp\n\ndef ask_audio(path, prompt, max_new_tokens, want_yes,\n              expected_sound, expected_silence):\n    # One question about one soundtrack, through the authors\'\n    # audio+text door (inference/infer.py, the branch for an\n    # audio file with no image and no video). The soundtrack comes\n    # out of the same file by the same moviepy step the video door\n    # uses; the question rides as text after the speech token; the\n    # blank zero image placeholders are passed exactly as that\n    # branch passes them. The feed proof prints before the answer\n    # is taken.\n    if not os.path.exists(path):\n        raise FeedProofError(f"feed file missing: {path}")\n    wav_path = extract_track(path)\n    proof = prove_track(wav_path, path, expected_sound, expected_silence)\n    speechs, speech_lengths, speech_chunks, speech_wavs = speech_tensors(\n        wav_path)\n    blank = torch.zeros(1, 3, 224, 224).to(\n        dtype=torch.bfloat16, device="cuda", non_blocking=True)\n    blank_highres = torch.zeros(1, 3, 224, 224).to(\n        dtype=torch.bfloat16, device="cuda", non_blocking=True)\n    images_arg = [blank]\n    images_highres_arg = [blank_highres]\n    image_sizes = [(224, 224)]\n    qs = DEFAULT_SPEECH_TOKEN + "\\n" + prompt\n    conv = conv_templates["qwen_1_5"].copy()\n    conv.append_message(conv.roles[0], qs)\n    conv.append_message(conv.roles[1], None)\n    full_prompt = conv.get_prompt()\n    input_ids = tokenizer_speech_token(\n        full_prompt, tokenizer, SPEECH_TOKEN_INDEX,\n        return_tensors="pt").unsqueeze(0).to("cuda")\n    proof["frames_total"] = None\n    proof["frames_sampled"] = None\n    proof["frames_kept"] = None\n    proof["feed"] = FEED_AUDIO_TEMPLATE.format(tokens=max_new_tokens)\n    print("FEED PROOF " + json.dumps(proof, sort_keys=True))\n    text, scores = run_generate(\n        input_ids,\n        dict(images=images_arg, images_highres=images_highres_arg,\n             image_sizes=image_sizes, modalities=["text"],\n             speech=speechs, speech_lengths=speech_lengths,\n             speech_chunks=speech_chunks, speech_wav=speech_wavs),\n        conv, max_new_tokens, want_yes)\n    yp = yes_probability(scores) if want_yes else None\n    if torch.cuda.is_available():\n        torch.cuda.empty_cache()\n    return text, proof, yp\n\ndef ask(job, prompt, max_tokens):\n    path = os.path.join(CLIPS, job["file"])\n    want_yes = prompt.rstrip().endswith("Answer yes or no.")\n    if job["arm"] == "audio_alone":\n        return ask_audio(path, prompt, max_tokens, want_yes,\n                         job["expected_sound"], job["expected_silence"])\n    return ask_video(path, prompt, max_tokens, want_yes,\n                     job["expected_sound"], job["expected_silence"])\n\n# --- Sanity check: can the model answer at all, through the real door? -----------\n# OLA\'s documented doors all carry media, so this gate rides the\n# full video path on the smallest clip in the folder, the beep\n# probe, with an instruction whose answer must contain the word\n# ready. It exercises the whole chain (decord, moviepy, both\n# speech encoders, the vision tower, generate, decode) before the\n# battery spends 87 asks on it. A wrong word, an empty answer, or\n# an error all stop the run here, loudly. Both modes run it.\nSANITY_CLIP = "/content/clips/probe_tone.mp4"\ntry:\n    sraw, _sproof, _syp = ask_video(SANITY_CLIP,\n                                    "Reply with the word ready.",\n                                    300, False, True, False)\n    print("SANITY RAW OUTPUT:", repr(sraw))\n    if not sraw:\n        print("WARNING: the sanity check came back EMPTY.")\n        print("STOP. Show this output to Muse.")\n        sys.exit(5)\n    if "ready" not in sraw.lower():\n        print("WARNING: the sanity answer does not contain the word ready.")\n        print("The ask path ran, but the model did not follow the one")\n        print("instruction it was given. STOP. Show this output to Muse.")\n        sys.exit(5)\n    print("sanity check passed: the model answered through the full")\n    print("video and sound path, and the word ready is in its answer.")\nexcept SystemExit:\n    raise\nexcept Exception as e:\n    print("WARNING: the sanity check FAILED with an error:")\n    print(str(e))\n    print("STOP. Show this output to Muse.")\n    sys.exit(5)\n\nif MODE == "check":\n    print()\n    print("check mode done: proof, download, ladder, and sanity all passed.")\n    print("The equal battery was not run. Run Step 4 for the tests.")\n    sys.exit(0)\n\n# --- Equal battery ---------------------------------------------------------\n# The repaired JSON question, injected at build time from\n# pilot2/run_eggs.py, character for character (444 characters).\nPROMPT_JSON = \'You are given a short first-person video clip. Watch and listen to the whole clip.\\nRespond with ONLY a JSON object in exactly this form, no other text: {"seen": "<the main things happening that you can SEE, short>", "heard": "<the main things happening that you can HEAR, short; write the word none if you hear no sound at all>", "sound_start_s": <number: the second at which the main sound event starts, or the word null if there is no sound>}\'\nassert len(PROMPT_JSON) == 444, f"prompt drifted: {len(PROMPT_JSON)} chars"\n\nprint("FEED SETTINGS, video asks:", FEED_VIDEO_TEMPLATE.format(\n    kept="N", sampled=64, total="T", tokens="M"))\nprint("FEED SETTINGS, sound-alone asks:", FEED_AUDIO_TEMPLATE.format(\n    tokens="M"))\nprint("MODEL:", MODEL_NAME, "in", LOAD_MODE, "form")\n\n# --- Test definitions: the equal battery (Series One, completed) -----------------\n# The plain sounds question, P3, injected at build time from\n# pilot2/swap/run_swap.py, character for character (143 characters).\nPROMPT_P3 = "What sounds can you hear in this clip? Describe only what is audible, without guessing from the pictures. If the audio is silent, say \'silent\'."\nassert len(PROMPT_P3) == 143, f"P3 drifted: {len(PROMPT_P3)} chars"\n\n# The transcription door for the bed test. New for this battery,\n# and the same words in all four equal-battery runners: bed\n# validity is judged per prompt, and this door gives every model\n# the baseline check in its strongest form.\nTRANSCRIPT_PROMPT = "Write down any words spoken in this clip. If no words are spoken, say \'no words\'."\n\nSWAP_FILES = {\n    "silence": "swap_silence.mp4",\n    "speech": "swap_speech.mp4",\n    "beeps": "swap_beeps.mp4",\n    "water": "swap_water.mp4",\n    "vacuum": "swap_vacuum.mp4",\n    "sizzle": "swap_sizzle.mp4",\n    "mixed": "swap_mixed.mp4",\n}\n\nREAL_CLIPS = [("vacuum", "P02_07_5.mp4"), ("jar", "P32_01_54.mp4"),\n              ("eggs", "own-01.mp4")]\n\nMATCHED_FILES = [("matched_speech", "matched_speech.mp4"),\n                 ("matched_speech_beeps", "matched_speech_beeps.mp4"),\n                 ("matched_speech_vacuum", "matched_speech_vacuum.mp4")]\n\ndef base_job(test, sound, file, arm, version, expected_sound):\n    return {"test": test, "sound": sound, "file": file, "arm": arm,\n            "version": version,\n            "expected_sound": bool(expected_sound),\n            "expected_silence": not bool(expected_sound)}\n\nJOBS = []\n\n# Part A: the repaired JSON question on the three real clips and\n# all seven swap files, two repeats each (20 asks).\nfor sound, file in REAL_CLIPS:\n    for rep in (1, 2):\n        job = base_job("partA_series_one", sound, file,\n                       "audio_with_video", "original", True)\n        JOBS.append((job, "open", "open_json", sound, PROMPT_JSON,\n                     rep, 300))\nfor sound in ("silence", "speech", "beeps", "water", "vacuum",\n              "sizzle", "mixed"):\n    for rep in (1, 2):\n        job = base_job("partA_series_one", sound, SWAP_FILES[sound],\n                       "audio_with_video", "original_swap_still",\n                       sound != "silence")\n        JOBS.append((job, "open", "open_json", sound, PROMPT_JSON,\n                     rep, 300))\n\n# Part A2: the same question on the three real clips muted and\n# blacked out, two repeats each (12 asks). The muted job declares\n# silence; the blackout job still carries the sound.\nfor sound, file in REAL_CLIPS:\n    for version, expected in (("muted", False), ("blackout", True)):\n        for rep in (1, 2):\n            job = base_job("partA2_muted_blackout", sound,\n                           f"equal_{version}_{file}",\n                           "audio_with_video", version, expected)\n            JOBS.append((job, "open", "open_json", sound, PROMPT_JSON,\n                         rep, 300))\n\n# Part B: the plain sounds question on the three matched bed\n# files, two repeats each (6 asks). The first file is the\n# baseline, the sentence alone; this prompt\'s bed rows are read\n# only if this prompt\'s baseline returned the sentence.\nfor sound, file in MATCHED_FILES:\n    for rep in (1, 2):\n        job = base_job("partB_matched_bed", sound, file,\n                       "audio_with_video", "original", True)\n        JOBS.append((job, "direct", "plain_sounds", "speech",\n                     PROMPT_P3, rep, 128))\n\n# Part B door: the transcription question on the same three\n# matched files, one repeat each (3 asks). Same reading rule:\n# the door\'s bed rows are read only if the door\'s baseline\n# returned the sentence.\nfor sound, file in MATCHED_FILES:\n    job = base_job("partB_door_transcription", sound, file,\n                   "audio_with_video", "original", True)\n    JOBS.append((job, "direct", "transcription", "speech",\n                 TRANSCRIPT_PROMPT, 1, 128))\n\nprint(f"equal battery jobs: {len(JOBS)} asks "\n      f"(part A: 20, part A2: 12, part B: 6, part B door: 3)")\n\nFIELDS = ["test", "sound", "file", "arm", "version", "question_form",\n          "question_label", "target", "prompt", "repeat",\n          "target_start_s", "expected_sound", "expected_silence",\n          "frame_ffmpeg_rc", "audio_ffmpeg_rc", "track_duration_s",\n          "sample_rate", "sample_count", "track_rms_db",\n          "track_nonzero_share", "unit_count", "per_unit_rms_db",\n          "per_unit_padding_samples", "padding_samples_total",\n          "zero_chunk_count", "track_sha256", "track_copy_paths",\n          "source_file_sha256", "raw_answer", "yes_score",\n          "yes_score_status", "model", "feed"]\n\nYES_STATUS_EXPOSED = ("exposed: the probability of a yes token at the "\n                      "first generated position, read from the "\n                      "output_scores of the same greedy generate call "\n                      "that produced the answer; the prompt, the "\n                      "feed, and the decoding are unchanged")\nYES_STATUS_NOT_YESNO = ("not a yes/no ask: this door exposes scores, "\n                        "but this question form has no yes token "\n                        "to score")\nYES_STATUS_MISSING = ("yes/no ask, but no score was recorded for "\n                      "this row; the words or the error in "\n                      "raw_answer are the record")\n\nrows = []\nfailed = 0\n\ndef save_rows():\n    out_fields = FIELDS\n    for out_path in ("/content/drive/MyDrive/project-k-clips/equal_ola_results.csv",\n                     "/content/equal_ola_results.csv"):\n        try:\n            with open(out_path, "w", newline="") as fh:\n                w = csv.DictWriter(fh, fieldnames=out_fields)\n                w.writeheader()\n                w.writerows(rows)\n            print("saved:", out_path)\n        except Exception as e:\n            print("could not save", out_path, "-", str(e)[:120])\n\nfor job, form, label, target, prompt, rep, max_tokens in JOBS:\n    print("=" * 70)\n    print(f"{job[\'test\']} - sound={job[\'sound\']} arm={job[\'arm\']} "\n          f"version={job[\'version\']} form={form}/{label} "\n          f"target={target} repeat={rep}")\n    want_yes = prompt.rstrip().endswith("Answer yes or no.")\n    raw_text = ""\n    error = ""\n    proof = None\n    yp = None\n    try:\n        raw, proof, yp = ask(job, prompt, max_tokens)\n        if not raw.strip():\n            error = "ERROR: empty answer returned"\n            failed += 1\n            print("WARNING: this ask came back EMPTY. Recorded as an error,")\n            print("never as silence.")\n            raw_text = error\n        else:\n            raw_text = raw\n            print("RAW ANSWER:")\n            print(raw)\n    except FeedProofError as e:\n        print()\n        print("!" * 70)\n        print("FEED PROOF HARD STOP. The feed for this ask failed its proof:")\n        print(str(e))\n        print("No answer was taken for this ask, and the run stops here.")\n        print("Rows completed before this ask are saved now. Show this")\n        print("whole output to Muse.")\n        print("!" * 70)\n        save_rows()\n        sys.exit(7)\n    except Exception as e:\n        error = "ERROR: " + traceback.format_exc()\n        failed += 1\n        print(f"WARNING: this ask FAILED with an error. The full error is")\n        print("saved in the CSV column raw_answer.")\n        raw_text = error\n        proof = proof or {}\n    p = proof or {}\n    if want_yes:\n        if yp is not None:\n            ys, yss = f"{yp:.4f}", YES_STATUS_EXPOSED\n        else:\n            ys, yss = "", YES_STATUS_MISSING\n    else:\n        ys, yss = "n/a", YES_STATUS_NOT_YESNO\n    rows.append({\n        "test": job["test"], "sound": job["sound"], "file": job["file"],\n        "arm": job["arm"], "version": job["version"],\n        "question_form": form, "question_label": label, "target": target,\n        "prompt": prompt, "repeat": rep,\n        "target_start_s": job.get("target_start_s", ""),\n        "expected_sound": job["expected_sound"],\n        "expected_silence": job["expected_silence"],\n        "frame_ffmpeg_rc": p.get("frame_ffmpeg_rc", ""),\n        "audio_ffmpeg_rc": p.get("audio_ffmpeg_rc", ""),\n        "track_duration_s": p.get("track_duration_s", ""),\n        "sample_rate": p.get("sample_rate", ""),\n        "sample_count": p.get("sample_count", ""),\n        "track_rms_db": p.get("track_rms_db", ""),\n        "track_nonzero_share": p.get("track_nonzero_share", ""),\n        "unit_count": p.get("unit_count", ""),\n        "per_unit_rms_db": json.dumps(p.get("per_unit_rms_db", "")),\n        "per_unit_padding_samples": json.dumps(\n            p.get("per_unit_padding_samples", "")),\n        "padding_samples_total": p.get("padding_samples_total", ""),\n        "zero_chunk_count": p.get("zero_chunk_count", ""),\n        "track_sha256": p.get("track_sha256", ""),\n        "track_copy_paths": json.dumps(p.get("track_copy_paths", "")),\n        "source_file_sha256": p.get("source_file_sha256", ""),\n        "raw_answer": raw_text,\n        "yes_score": ys,\n        "yes_score_status": yss,\n        "model": (f"{MODEL_NAME} ({LOAD_MODE}) "\n                  "[venv py3.10, torch 2.1.2+cu121]"),\n        "feed": p.get("feed", ""),\n    })\n    if torch.cuda.is_available():\n        torch.cuda.empty_cache()\n\nif failed:\n    print()\n    print("!" * 70)\n    print(f"RUN PROBLEM: {failed} ask(s) failed or came back empty. Their")\n    print("raw words or full errors are in the CSV column raw_answer.")\n    print("This run is NOT clean evidence until those asks are explained.")\n    print("!" * 70)\nelse:\n    print()\n    print("Clean run: every ask answered, every feed proved in print first.")\n\nsave_rows()\nif failed:\n    sys.exit(6)\n'
with open("/content/ola_equal_run.py", "w") as fh:
    fh.write(RUNNER_SOURCE)
print("runner written:", "/content/ola_equal_run.py", len(RUNNER_SOURCE), "chars")

In [ ]:
# --- Check mode: proof, download, ladder, sanity ---------------------------------
# Runs the runner under the private environment's Python and streams
# its output here live. A nonzero exit code means the proof, the
# download, the ladder, or the sanity gate failed, and the run stops
# in this cell. The first check run downloads about 21 GB of model
# files; that takes a while and is normal.
import os, subprocess
VENV_PY = "/content/ola-venv/bin/python"
proc = subprocess.run([VENV_PY, "/content/ola_equal_run.py", "check"])
print()
print("check exit code:", proc.returncode)
if proc.returncode != 0:
    print("!" * 72)
    print("STOP. THE CHECK RUN FAILED. The runner's own output above says")
    print("where: a proof line, the download, a ladder rung, or the")
    print("sanity question. Do not run Step 4. Show this output to Muse.")
    print("!" * 72)
    raise SystemExit("STOP: runner check failed")
print("check passed: proof lines right, model loaded, sanity answered.")
print("Go on to Step 4.")

## Step 4 · Run the equal battery

The next cell runs the runner in **battery mode**: proof and
ladder again, sanity again, then the 41 asks of the equal
battery, the completed Series One battery, identical in all
four equal-battery notebooks. Every ask prints its FEED PROOF
line before the model's answer is taken. Watch for it: that
line is the discipline that makes the answers evidence.

**How this model is asked.** the authors' own inference path: frames sampled across the whole clip (their selection check keeps a varying count, recorded per ask) and the whole soundtrack through the model's two audio encoders at once, Whisper large-v3 for speech and BEATs for sound events. Greedy decoding is the one deliberate change from the authors' demo. Decoding is greedy
everywhere. Token caps per ask are the job list's own: 300 for
the JSON question, 128 for the plain sounds and transcription
questions.

**Part A (20 asks).** The repaired JSON question, word for word
the Series One question, on the three real clips (vacuum, jar,
eggs) and all seven swap files, two repeats each.

**Part A2 (12 asks).** The same question on the three real
clips in the two versions Step 2 built: muted (the picture with
no sound, so any sound the answer reports is invented or
carried by the picture) and blackout (the sound with a black
picture, so the picture cannot help or interfere), two repeats
each.

**Part B (6 asks).** The plain sounds question (the swap
study's P3: "What sounds can you hear in this clip? Describe
only what is audible, without guessing from the pictures. If
the audio is silent, say 'silent'.") on the three matched bed
files, two repeats each.

**Part B door (3 asks).** "Write down any words spoken in this
clip. If no words are spoken, say 'no words'." on the same
three matched files, one repeat each.

**How the bed is read.** Per prompt, never pooled. Each
prompt's bed rows are read only if that same prompt's baseline
(the sentence alone, `matched_speech.mp4`) returned the
sentence. If the plain sounds baseline fails and the
transcription baseline returns the sentence, the bed is read
through the door, and the reverse likewise. Every row stays in
the CSV either way; the rule is applied at write-up time, not
by deleting rows. This replaces the Series One bed design,
where two of three bed tests came out void for want of a
passing baseline: here every model gets the baseline check in
both forms, including its strongest one.

**Hard stops and warnings.** A feed that fails its proof stops
the whole run on the spot (exit code 7), with the rows
completed so far saved first. An ask that errors or comes back
empty is saved with its full error text in `raw_answer` and a
WARNING on screen, never as silence, and the run ends with the
RUN PROBLEM banner (exit code 6). Results save to `equal_ola_results.csv` in
the Drive clips folder, one row per ask, with the full
feed-proof numbers on every row, in the same 33-column schema
as the Series Two results. The yes-score columns read n/a on
every row: none of these questions is a yes/no ask.

In [ ]:
# --- Battery mode: the equal battery tests --------------------------------
# Same runner, same private Python, streamed live. The runner saves
# the CSV itself, to the Drive clips folder and to /content. A
# nonzero exit code means the pre-flight gates failed up front, a
# feed proof hard-stopped the run, or the run finished with broken
# asks (the banners above say which).
import os, subprocess
VENV_PY = "/content/ola-venv/bin/python"
proc = subprocess.run([VENV_PY, "/content/ola_equal_run.py", "battery"])
print()
print("battery exit code:", proc.returncode)
if proc.returncode != 0:
    print("!" * 72)
    print("STOP. THE BATTERY RUN DID NOT FINISH CLEAN. Read the runner's")
    print("output above: a gate failed, a feed proof hard-stopped the run,")
    print("or some asks are broken and marked in the CSV. Show it to Muse.")
    print("!" * 72)
    raise SystemExit("STOP: battery run not clean")
print("battery done: every ask answered, every feed proved, CSV saved.")

## Closing · How to read the answers

Read this run against this model's own Series One and Series
Two answers, and against the other three models' equal-battery
runs. The battery is identical across the four notebooks: same
files, same questions byte for byte, same repeats. A difference
is a model difference or a feed difference, and the feed
differences are written down, not hidden. This model's feed:
the authors' own inference path: frames sampled across the whole clip (their selection check keeps a varying count, recorded per ask) and the whole soundtrack through the model's two audio encoders at once, Whisper large-v3 for speech and BEATs for sound events. Greedy decoding is the one deliberate change from the authors' demo.

- **Part A against Part A2.** Clean minus muted is what the
  sound adds to the answer. Blackout is what the sound does
  when the picture cannot carry the answer: a model that names
  the sound on the clean clip but not on the blackout was
  reading the picture, not hearing the sound.
- **Part B under the baseline rule.** Read each prompt's bed
  rows only where that prompt's baseline returned the sentence
  (Step 4 states the rule). A bed that bends the transcription
  while the report denies the bed is the bed test's signature
  finding, and it only counts where the baseline passed.
- **The fit.** The `model` column of every row names the rung
  that loaded. A squeezed fit's answers are that fit's answers.
- **Scope.** Every claim stays capped at this model, this
  feed, these files, this date.

When you finish, the file `equal_ola_results.csv` in your Drive is the
record. Send it back, and the four equal-battery runs get
written up side by side.